# Chapter 1 Preprocessing Runbook

This notebook is a thin orchestration and QC layer for the canonical Chapter 1 preprocessing pipeline in this repository.

- It reads standardized upstream ASIC artifacts from explicit artifact paths.
- It uses the shared JSON run config that is also used by the CLI.
- It calls the package pipeline once, then displays the cohort, instance, label, and model-ready outputs stage by stage.
- It writes final outputs into this repo's artifact directory.


## Configuration

Edit the shared run config file before executing the notebook end to end. The default repo config points at the standardized ASIC artifact root in `icu-data-platform`.


In [1]:
from pathlib import Path

RUN_CONFIG_PATH = Path("config") / "ch1_run_config.json"


In [2]:
import sys
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "chapter1_mortality_decomposition").exists():
    if (PROJECT_ROOT.parent / "src" / "chapter1_mortality_decomposition").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repo root or the notebooks directory.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from chapter1_mortality_decomposition.artifacts import load_chapter1_inputs
from chapter1_mortality_decomposition.pipeline import (
    build_chapter1_dataset,
    write_chapter1_dataset,
)
from chapter1_mortality_decomposition.run_config import load_chapter1_run_config

pd.set_option("display.max_colwidth", 160)

run_config = load_chapter1_run_config(PROJECT_ROOT / RUN_CONFIG_PATH)
config = run_config.to_chapter1_config()
input_dir = run_config.input_dir
output_dir = run_config.output_dir

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))
display(Markdown(f"**Run config:** `{run_config.run_config_path}`"))
display(
    pd.DataFrame(
        [
            {"setting": "input_dir", "value": str(input_dir)},
            {"setting": "input_format", "value": run_config.input_format},
            {"setting": "output_dir", "value": str(output_dir)},
            {"setting": "output_format", "value": run_config.output_format},
            {"setting": "feature_set_config_path", "value": str(run_config.feature_set_config_path)},
            {"setting": "horizons_hours", "value": list(run_config.horizons_hours)},
            {"setting": "min_required_core_groups", "value": run_config.min_required_core_groups},
            {
                "setting": "mech_vent_qc_artifact",
                "value": str(input_dir / "qc" / f"mech_vent_ge_24h_stay_level.{run_config.input_format}"),
            },
        ]
    )
)


**Project root:** `/Users/joanameyer/repository/1-mortality-decomposition`

**Run config:** `/Users/joanameyer/repository/1-mortality-decomposition/config/ch1_run_config.json`

,setting,value
0,input_dir,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized
1,input_format,csv
2,output_dir,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1
3,output_format,csv
4,feature_set_config_path,/Users/joanameyer/repository/1-mortality-decomposition/config/ch1_feature_sets.json
5,horizons_hours,"[8, 16, 24, 48, 72]"
6,min_required_core_groups,3
7,mech_vent_qc_artifact,/Users/joanameyer/repository/icu-data-platform/artifacts/asic_harmonized/qc/mech_vent_ge_24h_stay_level.csv


In [3]:
if not input_dir.exists():
    raise FileNotFoundError(
        f"Input directory does not exist: {input_dir}. Update {run_config.run_config_path}."
    )

inputs = load_chapter1_inputs(input_dir=input_dir, input_format=run_config.input_format)

artifact_overview = pd.DataFrame(
    [
        {"table": "static_harmonized", "rows": inputs.static_harmonized.shape[0], "columns": inputs.static_harmonized.shape[1]},
        {"table": "dynamic_harmonized", "rows": inputs.dynamic_harmonized.shape[0], "columns": inputs.dynamic_harmonized.shape[1]},
        {"table": "block_index", "rows": inputs.block_index.shape[0], "columns": inputs.block_index.shape[1]},
        {"table": "blocked_dynamic_features", "rows": inputs.blocked_dynamic_features.shape[0], "columns": inputs.blocked_dynamic_features.shape[1]},
        {"table": "stay_block_counts", "rows": inputs.stay_block_counts.shape[0], "columns": inputs.stay_block_counts.shape[1]},
        {"table": "mech_vent_stay_level_qc", "rows": inputs.mech_vent_stay_level_qc.shape[0], "columns": inputs.mech_vent_stay_level_qc.shape[1]},
    ]
)
display(Markdown("## Loaded Upstream Artifacts"))
display(artifact_overview)


## Loaded Upstream Artifacts

,table,rows,columns
0,static_harmonized,80,16
1,dynamic_harmonized,105165,107
2,block_index,3250,6
3,blocked_dynamic_features,3250,621
4,stay_block_counts,80,9
5,mech_vent_stay_level_qc,80,6


In [4]:
dataset = build_chapter1_dataset(inputs, config=config)


In [5]:
display(Markdown("## Cohort QC"))
display(pd.DataFrame([
    {
        "retained_hospitals": dataset.cohort.retained_hospitals.shape[0],
        "retained_stays": dataset.cohort.table.shape[0],
    }
]))
display(Markdown("### Site-level eligibility"))
display(dataset.cohort.site_eligibility[["hospital_id", "site_included_ch1", "usable_core_vital_group_count", "exclusion_reason"]])
display(Markdown("### Stay-level exclusions"))
display(dataset.cohort.stay_exclusion_summary_by_hospital)
display(Markdown("### Canonical cohort summary"))
display(dataset.cohort_summary)
display(Markdown("### Verification checks"))
display(dataset.verification_summary)


## Cohort QC

,retained_hospitals,retained_stays
0,4,35


### Site-level eligibility

,hospital_id,site_included_ch1,usable_core_vital_group_count,exclusion_reason
0,asic_UK00,False,4,missing/unusable icu_mortality at site level
1,asic_UK01,False,0,missing/unusable icu_mortality at site level; insufficient core-vitals coverage
2,asic_UK02,True,4,<NA>
3,asic_UK03,False,3,missing/unusable icu_mortality at site level
4,asic_UK04,True,4,<NA>
5,asic_UK06,False,0,insufficient core-vitals coverage
6,asic_UK07,True,4,<NA>
7,asic_UK08,True,4,<NA>


### Stay-level exclusions

,hospital_id,before_site_level_exclusion,after_site_level_exclusion,after_mech_vent_ge_24h_qc_exclusion,after_no_dynamic_data_exclusion,after_missing_readmission_exclusion,after_readmission_flagged_exclusion,after_missing_icu_mortality_exclusion,final_retained_stays,excluded_missing_mech_vent_ge_24h_qc_stays,excluded_mech_vent_ge_24h_qc_failed_stays,excluded_no_dynamic_data_stays,excluded_missing_readmission_stays,excluded_readmission_flagged_stays,excluded_missing_icu_mortality_stays
0,asic_UK00,10,0,0,0,0,0,0,0,0,0,0,0,0,0
1,asic_UK01,10,0,0,0,0,0,0,0,0,0,0,0,0,0
2,asic_UK02,10,10,10,10,10,10,10,10,0,0,0,0,0,0
3,asic_UK03,10,0,0,0,0,0,0,0,0,0,0,0,0,0
4,asic_UK04,10,10,9,9,9,9,9,9,0,1,0,0,0,0
5,asic_UK06,10,0,0,0,0,0,0,0,0,0,0,0,0,0
6,asic_UK07,10,10,10,10,10,9,9,9,0,0,0,0,1,0
7,asic_UK08,10,10,7,7,7,7,7,7,0,3,0,0,0,0


### Canonical cohort summary

,summary_group,metric,horizon_h,value
0,cohort,input_hospitals,<NA>,8
1,cohort,retained_hospitals,<NA>,4
2,cohort,input_stays,<NA>,80
3,cohort,retained_stays,<NA>,35
4,instances,valid_prediction_instances_total,<NA>,8390
5,instances,valid_prediction_instances,8,1678
6,instances,valid_prediction_instances,16,1678
7,instances,valid_prediction_instances,24,1678
8,instances,valid_prediction_instances,48,1678
9,instances,valid_prediction_instances,72,1678


### Verification checks

,check_id,passed,detail
0,no_excluded_hospital_contributes_retained_stays,True,All retained stays and valid prediction instances come from site-eligible hospitals.
1,no_mech_vent_failed_stay_retained,True,No retained stay fails the upstream mech_vent_ge_24h_qc contract.
2,no_missing_or_readmission_flagged_stay_retained,True,No retained stay has missing readmission or readmission == 1.
3,valid_instances_restricted_to_retained_stays,True,Valid prediction instances are computed only on retained stays.
4,proxy_label_counts_consistent_with_valid_instances,True,"Per-horizon labelable and unlabeled counts sum to the valid-instance counts, and usable label rows match the total labelable count."


In [6]:
display(Markdown("## Feature-set Configuration and Validation"))
display(dataset.feature_set_validation_summary[[
    "feature_set_name",
    "primary_feature_count",
    "extended_only_feature_count",
    "total_extended_feature_count",
    "available_in_blocked_schema_count",
    "missing_from_blocked_schema_count",
    "absent_from_retained_data_count",
    "missing_from_blocked_schema_features",
    "absent_from_retained_data_features",
]])


## Feature-set Configuration and Validation

,feature_set_name,primary_feature_count,extended_only_feature_count,total_extended_feature_count,available_in_blocked_schema_count,missing_from_blocked_schema_count,absent_from_retained_data_count,missing_from_blocked_schema_features,absent_from_retained_data_features
0,primary,31,15,46,31,0,0,[],[]
1,extended,31,15,46,46,0,0,[],[]


In [7]:
display(Markdown("## Valid Prediction Instances"))
display(dataset.valid_instances.counts_by_horizon)
display(dataset.valid_instances.exclusion_summary)

display(Markdown("## Proxy Horizon Labels"))
display(dataset.labels.summary_by_horizon)
display(dataset.labels.unlabeled_reason_summary)


## Valid Prediction Instances

,horizon_h,candidate_instances,valid_instances,excluded_instances
0,8,1704,1678,26
1,16,1704,1678,26
2,24,1704,1678,26
3,48,1704,1678,26
4,72,1704,1678,26


,horizon_h,exclusion_reason,instance_count
0,8,insufficient_core_vital_group_coverage_in_block,26
1,16,insufficient_core_vital_group_coverage_in_block,26
2,24,insufficient_core_vital_group_coverage_in_block,26
3,48,insufficient_core_vital_group_coverage_in_block,26
4,72,insufficient_core_vital_group_coverage_in_block,26


## Proxy Horizon Labels

,horizon_h,total_valid_prediction_instances,labelable_instances,positive_labels,negative_labels,unlabeled_instances
0,8,1678,1256,10,1246,422
1,16,1678,1244,20,1224,434
2,24,1678,1232,30,1202,446
3,48,1678,1194,58,1136,484
4,72,1678,1154,82,1072,524


,horizon_h,unlabeled_reason,instance_count
0,8,missing_required_fields,0
1,8,prediction_time_not_strictly_before_proxy_end,1
2,8,survivor_without_full_horizon_observation,21
3,8,non_survivor_proxy_end_not_within_horizon,400
4,8,unsupported_icu_mortality_code,0
5,16,missing_required_fields,0
6,16,prediction_time_not_strictly_before_proxy_end,1
7,16,survivor_without_full_horizon_observation,43
8,16,non_survivor_proxy_end_not_within_horizon,390
9,16,unsupported_icu_mortality_code,0


In [8]:
display(Markdown("## Model-ready Datasets"))
final_qc_rows = []
for feature_set_name, feature_set_dataset in dataset.feature_sets.items():
    display(Markdown(f"### {feature_set_name.title()} Feature Set"))
    display(feature_set_dataset.model_ready.readiness_summary)
    readiness = feature_set_dataset.model_ready.readiness_summary.set_index("metric")
    final_qc_rows.append(
        {
            "feature_set_name": feature_set_name,
            "model_ready_rows_total": int(feature_set_dataset.model_ready.table.shape[0]),
            "selected_feature_columns_total": readiness.at["selected_feature_columns_total", "value"],
            "configured_base_features_total": readiness.at["configured_base_features_total", "value"],
        }
    )
display(Markdown("### Primary vs Extended Summary"))
display(pd.DataFrame(final_qc_rows))


## Model-ready Datasets

### Primary Feature Set

,feature_set_name,metric,value,note
0,primary,model_ready_rows_total,6080,Rows after valid-instance selection and label availability filtering.
1,primary,configured_base_features_total,31,Configured base features for this Chapter 1 feature set.
2,primary,selected_feature_columns_total,186,Blocked dynamic feature columns selected by the Chapter 1 feature config.
3,primary,distinct_horizons_total,5,Configured prediction horizons represented in the model-ready table.
4,primary,configured_base_features_missing_from_blocked_schema,,Configured base features absent from the current blocked schema.
5,primary,configured_base_features_absent_from_retained_data,,Configured base features absent from current retained-hospital ASIC data.
6,primary,label_definition_in_model_ready_dataset,proxy_within_horizon_icu_mortality_v1,Current model-ready rows use the explicit ASIC proxy within-horizon label definition.


### Extended Feature Set

,feature_set_name,metric,value,note
0,extended,model_ready_rows_total,6080,Rows after valid-instance selection and label availability filtering.
1,extended,configured_base_features_total,46,Configured base features for this Chapter 1 feature set.
2,extended,selected_feature_columns_total,276,Blocked dynamic feature columns selected by the Chapter 1 feature config.
3,extended,distinct_horizons_total,5,Configured prediction horizons represented in the model-ready table.
4,extended,configured_base_features_missing_from_blocked_schema,,Configured base features absent from the current blocked schema.
5,extended,configured_base_features_absent_from_retained_data,,Configured base features absent from current retained-hospital ASIC data.
6,extended,label_definition_in_model_ready_dataset,proxy_within_horizon_icu_mortality_v1,Current model-ready rows use the explicit ASIC proxy within-horizon label definition.


### Primary vs Extended Summary

,feature_set_name,model_ready_rows_total,selected_feature_columns_total,configured_base_features_total
0,primary,6080,186,31
1,extended,6080,276,46


In [9]:
output_paths = write_chapter1_dataset(
    dataset,
    output_dir=output_dir,
    output_format=run_config.output_format,
)

display(Markdown("## Outputs Written"))
output_manifest = pd.DataFrame(
    {
        "artifact_key": list(output_paths.keys()),
        "path": [str(path) for path in output_paths.values()],
    }
)
display(output_manifest)

print(f"Saved {len(output_paths)} output tables to {output_dir}")


## Outputs Written

,artifact_key,path
0,cohort_chapter1_notes,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_notes.csv
1,cohort_chapter1_core_vital_group_coverage,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_core_vital_group_coverage.csv
2,cohort_chapter1_site_eligibility,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_site_eligibility.csv
3,cohort_chapter1_site_counts_summary,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_site_counts_summary.csv
4,cohort_chapter1_stay_exclusions,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_stay_exclusions.csv
5,cohort_chapter1_stay_exclusion_summary_by_hospital,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_stay_exclusion_summary_by_hospital.csv
6,cohort_chapter1_counts_by_hospital,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_counts_by_hospital.csv
7,cohort_chapter1_retained_hospitals,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_hospitals.csv
8,cohort_chapter1_retained_stays,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stays.csv
9,cohort_chapter1_retained_stay_table,/Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1/cohort/chapter1_retained_stay_table.csv


Saved 29 output tables to /Users/joanameyer/repository/1-mortality-decomposition/artifacts/chapter1
